**Latihan 1 (KNN Sederhana untuk Klasifikasi Suhu (Panas/Dingin))**

Import library yang dibutuhkan

In [12]:
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, classification_report

In [2]:
# 1. Buat Dataset kecil
data = {
    "temperature": [10, 25, 15, 20, 18, 20, 22, 24],
    "wind":        [0,  0,  5,  3,  7, 10,  5,  6],
    "label":       ["Dingin","Panas","Dingin","Panas","Dingin","Dingin","Panas","Panas"]
}

df = pd.DataFrame(data)
print("DATASET:\n", df)

DATASET:
    temperature  wind   label
0           10     0  Dingin
1           25     0   Panas
2           15     5  Dingin
3           20     3   Panas
4           18     7  Dingin
5           20    10  Dingin
6           22     5   Panas
7           24     6   Panas


In [3]:
# 2. Encode label (Panas/Dingin -> angka)
le = LabelEncoder()
df["label_encoded"] = le.fit_transform(df["label"])

In [4]:
# 3. Tentukan fitur (X) dan target (y)
X = df[["temperature", "wind"]]
y = df["label_encoded"]

In [5]:
# 4. Buat model KNN (K = 3)
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X, y)

KNeighborsClassifier(n_neighbors=3)

In [6]:
# 5. Mencoba prediksi data baru (misal suhu = 23, angin = 4)
data_baru = [[23, 4]]
prediksi = knn.predict(data_baru)
hasil = le.inverse_transform(prediksi)

print("\nPrediksi untuk data baru (suhu=23, angin=4):", hasil[0])


Prediksi untuk data baru (suhu=23, angin=4): Panas


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(


**Latihan 2 (confusion matrix + accuracy + precision + recall)**

In [8]:
# 1. Prediksi semua data
y_pred = knn.predict(X)

In [9]:
# 2. Confusion Matrix
cm = confusion_matrix(y, y_pred)
print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[4 0]
 [0 4]]


In [10]:
# Accuracy, Precision dan Recall
acc = accuracy_score(y, y_pred)
prec = precision_score(y, y_pred, average='binary')
rec = recall_score(y, y_pred, average='binary')

print("\nAccuracy :", acc)
print("Precision:", prec)
print("Recall   :", rec)


Accuracy : 1.0
Precision: 1.0
Recall   : 1.0


In [11]:
# Laporan lengkap
print("\nClassification Report:")
print(classification_report(y, y_pred, target_names=le.classes_))


Classification Report:
              precision    recall  f1-score   support

      Dingin       1.00      1.00      1.00         4
       Panas       1.00      1.00      1.00         4

    accuracy                           1.00         8
   macro avg       1.00      1.00      1.00         8
weighted avg       1.00      1.00      1.00         8



**Latihan 3 (Prediksi Cuaca dengan KNN (Menggunakan Dataset Weather))**

In [16]:
# 1. Load dataset
df_weather = pd.read_csv('/content/drive/MyDrive/Machine Learning/Praktikum 10 | KNN/data/weather_classification_data (1).csv')
print("5 Data Teratas:")
display(df_weather.head())

5 Data Teratas:


,Temperature,Humidity,Wind Speed,Precipitation (%),Cloud Cover,Atmospheric Pressure,UV Index,Season,Visibility (km),Location,Weather Type
0,14.0,73,9.5,82.0,partly cloudy,1010.82,2,Winter,3.5,inland,Rainy
1,39.0,96,8.5,71.0,partly cloudy,1011.43,7,Spring,10.0,inland,Cloudy
2,30.0,64,7.0,16.0,clear,1018.72,5,Spring,5.5,mountain,Sunny
3,38.0,83,1.5,82.0,clear,1026.25,7,Spring,1.0,coastal,Sunny
4,27.0,74,17.0,66.0,overcast,990.67,1,Winter,2.5,mountain,Rainy


In [18]:
# 2. Encode fitur kategori: Cloud Cover, Season, Location, Weather Type
label_columns = ["Cloud Cover", "Season", "Location", "Weather Type"]

encoders = {}
for col in label_columns:
    enc = LabelEncoder()
    df_weather[col] = enc.fit_transform(df_weather[col])
    encoders[col] = enc

In [19]:
# 3. Tentukan fitur & target
X = df_weather.drop("Weather Type", axis=1)  # semua fitur kecuali label
y = df_weather["Weather Type"]

In [24]:
# 4. Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [26]:
# 5. Buat model KNN (K = 4)
knn_weather = KNeighborsClassifier(n_neighbors=4)
knn_weather.fit(X_train, y_train)

KNeighborsClassifier(n_neighbors=4)

In [22]:
# 6. Prediksi data test
y_pred_weather = knn_weather.predict(X_test)

In [23]:
# 7. Evaluasi model
print("\n=== CONFUSION MATRIX ===")
print(confusion_matrix(y_test, y_pred_weather))

print("\n=== AKURASI ===")
print(accuracy_score(y_test, y_pred_weather))

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred_weather))


=== CONFUSION MATRIX ===
[[817  78  29  31]
 [ 60 883  14  25]
 [ 27  17 976  13]
 [ 61  51  31 847]]

=== AKURASI ===
0.8896464646464647

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

           0       0.85      0.86      0.85       955
           1       0.86      0.90      0.88       982
           2       0.93      0.94      0.94      1033
           3       0.92      0.86      0.89       990

    accuracy                           0.89      3960
   macro avg       0.89      0.89      0.89      3960
weighted avg       0.89      0.89      0.89      3960



In [27]:
# 6. Buat Dataframe baru untuk prediksi
cuaca_baru = {
    "Temperature": 20,
    "Humidity": 70,
    "Wind Speed": 5,
    "Precipitation (%)": 40,
    "Cloud Cover": "overcast",
    "Atmospheric Pressure": 1005,
    "UV Index": 3,
    "Season": "Winter",
    "Visibility (km)": 5,
    "Location": "inland"
}

In [28]:
# 7. Ubah ke DataFrame agar sesuai format input model
df_input = pd.DataFrame([cuaca_baru])

In [ ]:
# 8. Encoding kolom kategorikal menggunakan encoder dari dataset asli
for kolom in df_input.columns:
    if df_input[kolom].dtype == "object":
        df_input[kolom] = encoders[kolom].transform(df_input[kolom])

In [31]:
# 9. Gunakan model KNN yang sudah dilatih sebelumnya

# Pastikan kolom kategorikal di df_input di-encode sebelum prediksi
for kolom in df_input.columns:
    if df_input[kolom].dtype == "object":
        if kolom in encoders:
            df_input[kolom] = encoders[kolom].transform(df_input[kolom])
        else:
            print(f"Peringatan: Encoder untuk kolom '{kolom}' tidak ditemukan. Tidak dapat melakukan encoding.")

prediksi_encoded = knn_weather.predict(df_input)[0]

In [ ]:
predicted_weather_type = encoders['Weather Type'].inverse_transform([prediksi_encoded])
print("Prediksi Cuaca:", predicted_weather_type[0])

In [32]:
# Kembalikan hasil prediksi ke label asli (Rainy, Sunny, Cloudy)
hasil_prediksi = encoders["Weather Type"].inverse_transform([prediksi_encoded])[0]

print("Hasil prediksi kondisi cuaca untuk data baru:", hasil_prediksi)

Hasil prediksi kondisi cuaca untuk data baru: Cloudy
